# Assignment 6 — Class imbalance, hypothesis tests, and Naive Bayes

**SVNIT — Department of Artificial Intelligence**

**Question sheet:** `LAB ASSIGNMENT-6.pdf`

Two separate problems:

1. **Credit-card fraud** (`../data/creditcard.csv`, 284,807 transactions). Class `1` is fraud and is about 0.17% of the rows. Parts A–D cover EDA, why imbalance breaks accuracy, under-sampling, over-sampling / SMOTE, and a hypothesis test used as a feature check.
2. **SMS spam** (`../data/sms_spam.tsv`). Clean the text, turn it into numbers, then compare Bernoulli Naive Bayes with Multinomial Naive Bayes.

Plots are saved to `../plots/assignment_6/` and shown under the code that creates them. Every code cell is commented so the *reason* for each step is in the notebook.


In [7]:
import string
import warnings
from pathlib import Path

import matplotlib
# Agg writes plots to files. That avoids huge inline PNGs in the notebook.
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)


def find_project_root():
    """Walk upwards until we find the data/ folder, so the notebook runs
    whether the kernel's working directory is the repo or the Assignment 6 folder."""
    starts = [Path.cwd().resolve()]
    for key in ("__vsc_ipynb_file__", "__session__"):
        val = globals().get(key)
        if val:
            starts.append(Path(val).resolve().parent)
    seen = set()
    for start in starts:
        for p in [start, *start.parents]:
            if p in seen:
                continue
            seen.add(p)
            if (p / "data" / "creditcard.csv").exists():
                return p
    raise FileNotFoundError("Could not find data/creditcard.csv")


ROOT = find_project_root()
DATA = ROOT / "data"
PLOTS = ROOT / "plots" / "assignment_6"
PLOTS.mkdir(parents=True, exist_ok=True)


def section(title):
    print("\n" + "=" * 72)
    print(f"  {title}")
    print("=" * 72)


def save_plot(name):
    path = PLOTS / name
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"Saved {path.relative_to(ROOT)}")


def fraud_scores(y_true, y_pred, name):
    """Scores for the fraud class only (label 1).

    Accuracy is printed too, but it is the wrong headline metric here:
    a model that always says "not fraud" is already ~99.8% accurate.
    Precision = of the alarms we raised, how many were real fraud.
    Recall    = of the real frauds, how many we caught.
    F1        = the harmonic mean of those two. It is low if either is low.
    """
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }
    return row


print("Project root:", ROOT)


Project root: /Users/ojas/Desktop/ML Lab


## Problem 1 — Credit card fraud

The file is the well-known European card-transaction dataset. `Time` is seconds since the first transaction. `V1`–`V28` are already principal components of the original (confidential) features. `Amount` is the transaction value. `Class` is 0 for a normal payment and 1 for fraud.


In [8]:
section("Part A: EDA")

fraud = pd.read_csv(DATA / "creditcard.csv")

# Nothing should be missing in this file. We check anyway, because a later
# scaler or hypothesis test would silently drop or crash on NaN.
print(f"Rows: {len(fraud):,}    Columns: {fraud.shape[1]}")
print("Columns:", list(fraud.columns))
print(f"Missing values: {int(fraud.isna().sum().sum())}")

class_counts = fraud["Class"].value_counts().sort_index()
n_legit = int(class_counts[0])
n_fraud = int(class_counts[1])
fraud_rate = n_fraud / len(fraud)

print(f"\nLegitimate (0): {n_legit:,}")
print(f"Fraud        (1): {n_fraud:,}")
print(f"Fraud rate:        {100 * fraud_rate:.3f}%")
print("\nAmount by class (fraud amounts are not always the huge ones):")
print(fraud.groupby("Class")["Amount"].describe().round(2).to_string())

# Two views of the same fact. The pie makes the imbalance obvious.
# The bar uses a log scale so the 492 frauds are still visible next to 284,315 legitimate rows.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].pie(
    [n_legit, n_fraud],
    labels=["Legitimate", "Fraud"],
    autopct="%1.2f%%",
    colors=["steelblue", "crimson"],
    startangle=90,
)
axes[0].set_title("Class share")
axes[1].bar(["Legitimate", "Fraud"], [n_legit, n_fraud], color=["steelblue", "crimson"])
axes[1].set_yscale("log")
axes[1].set_ylabel("Count (log scale)")
axes[1].set_title("Class counts")
save_plot("class_distribution.png")

# Amount is heavily skewed. A log x-axis shows whether fraud sits in a different
# dollar range from normal spending. (log1p so a zero-amount row still plots.)
plt.figure(figsize=(8, 4.5))
for label, color, name in [(0, "steelblue", "Legitimate"), (1, "crimson", "Fraud")]:
    values = fraud.loc[fraud["Class"] == label, "Amount"]
    plt.hist(np.log1p(values), bins=40, alpha=0.55, color=color, label=name, density=True)
plt.xlabel("log(1 + Amount)")
plt.ylabel("Density")
plt.title("Transaction amount by class")
plt.legend()
save_plot("amount_by_class.png")



  Part A: EDA
Rows: 284,807    Columns: 31
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
Missing values: 0

Legitimate (0): 284,315
Fraud        (1): 492
Fraud rate:        0.173%

Amount by class (fraud amounts are not always the huge ones):
             count     mean      std    min    25%     50%      75%         max
Class                                                                          
0     284,315.0000  88.2900 250.1100 0.0000 5.6500 22.0000  77.0500 25,691.1600
1         492.0000 122.2100 256.6800 0.0000 1.0000  9.2500 105.8900  2,125.8700
Saved plots/assignment_6/class_distribution.png
Saved plots/assignment_6/amount_by_class.png


**Class balance and amount**

![Class distribution](../plots/assignment_6/class_distribution.png)

![Amount by class](../plots/assignment_6/amount_by_class.png)


In [9]:
section("Part B: Why imbalance is a problem")

# A useless rule — "always predict legitimate" — gets this accuracy.
# That is the trap. Accuracy rewards the majority class.
always_legit_accuracy = n_legit / len(fraud)
print(f"Accuracy of predicting every row as legitimate: {100 * always_legit_accuracy:.3f}%")
print(f"Recall of that rule on fraud: 0%  (it never catches a fraud)")
print()
print("What goes wrong if we train on the raw data:")
print("  1. The loss is dominated by the 284,315 legitimate rows.")
print("  2. The model can lower the loss by predicting 0 almost everywhere.")
print("  3. Accuracy still looks excellent, while fraud recall is near zero.")
print("  4. The decision threshold 0.5 is then much too high for a rare event.")
print()
print("Fix used below: change the TRAINING set so the two classes have equal")
print("counts, then score the model on the ORIGINAL imbalanced test set.")
print("We never resample the test set. The test set must look like real traffic.")

# Same split for every sampling method, so the comparison is fair.
# stratify=y keeps the 0.17% fraud rate in both sides.
feature_cols = [c for c in fraud.columns if c != "Class"]
X = fraud[feature_cols]
y = fraud["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# V1-V28 are already scaled PCA scores, but Time and Amount are not.
# Logistic regression uses the size of each coefficient, so we put every
# column on a common scale. Fit the scaler on train only.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"\nTrain rows: {len(y_train):,}   fraud in train: {int(y_train.sum()):,}")
print(f"Test rows:  {len(y_test):,}   fraud in test:  {int(y_test.sum()):,}")



  Part B: Why imbalance is a problem
Accuracy of predicting every row as legitimate: 99.827%
Recall of that rule on fraud: 0%  (it never catches a fraud)

What goes wrong if we train on the raw data:
  1. The loss is dominated by the 284,315 legitimate rows.
  2. The model can lower the loss by predicting 0 almost everywhere.
  3. Accuracy still looks excellent, while fraud recall is near zero.
  4. The decision threshold 0.5 is then much too high for a rare event.

Fix used below: change the TRAINING set so the two classes have equal
counts, then score the model on the ORIGINAL imbalanced test set.
We never resample the test set. The test set must look like real traffic.

Train rows: 227,845   fraud in train: 394
Test rows:  56,962   fraud in test:  98


In [10]:
section("Part C: Under-sampling, over-sampling, SMOTE")

# All three samplers change ONLY the training matrix.
# random_state=42 makes the random draws repeatable.
samplers = {
    # Throw away legitimate training rows until counts match the fraud count.
    # Fast, but most of the legitimate patterns are discarded.
    "Random under-sampling": RandomUnderSampler(random_state=42),
    # Copy fraud rows until they match the legitimate count.
    # Nothing new is learned; the same frauds are just repeated.
    "Random over-sampling": RandomOverSampler(random_state=42),
    # Invent new fraud rows on the line between a fraud and one of its
    # nearest fraud neighbours. Minority count is brought up to the majority count.
    "SMOTE": SMOTE(random_state=42),
}

# Baseline: train on the imbalanced data, no resampling.
# This is the "do nothing" model the sampling methods have to beat.
models = {
    "No resampling": (X_train_s, y_train.to_numpy()),
}
for name, sampler in samplers.items():
    X_res, y_res = sampler.fit_resample(X_train_s, y_train)
    models[name] = (X_res, y_res)
    print(f"{name:<24} train rows {len(y_res):>8,}   fraud {int((y_res == 1).sum()):>8,}")

print(f"\n{'Model':<24} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 68)
score_rows = []
for name, (X_res, y_res) in models.items():
    # Same classifier everywhere. The only thing that changes is the training mix.
    # max_iter is high because the over-sampled sets are large.
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_res, y_res)
    pred = clf.predict(X_test_s)
    row = fraud_scores(y_test, pred, name)
    score_rows.append(row)
    print(
        f"{name:<24} {100 * row['accuracy']:>9.2f}% {row['precision']:>10.3f} "
        f"{row['recall']:>10.3f} {row['f1']:>10.3f}"
    )

scores = pd.DataFrame(score_rows).set_index("model")
# "Best" is decided by fraud F1, not accuracy. Accuracy stays high for all of them.
best_name = scores["f1"].idxmax()
print(f"\nHighest fraud F1: {best_name}  (F1 = {scores.loc[best_name, 'f1']:.3f})")

plot_df = scores.reset_index().melt(
    id_vars="model", value_vars=["precision", "recall", "f1"],
    var_name="metric", value_name="score",
)
plt.figure(figsize=(9, 4.8))
sns.barplot(data=plot_df, x="model", y="score", hue="metric")
plt.ylim(0, 1)
plt.ylabel("Score on the fraud class")
plt.xlabel("")
plt.xticks(rotation=15, ha="right")
plt.title("Fraud precision, recall, and F1")
save_plot("sampling_comparison.png")



  Part C: Under-sampling, over-sampling, SMOTE
Random under-sampling    train rows      788   fraud      394
Random over-sampling     train rows  454,902   fraud  227,451
SMOTE                    train rows  454,902   fraud  227,451

Model                      Accuracy  Precision     Recall         F1
--------------------------------------------------------------------
No resampling                99.91%      0.827      0.633      0.717
Random under-sampling        96.02%      0.038      0.918      0.074
Random over-sampling         97.57%      0.061      0.918      0.115
SMOTE                        97.41%      0.058      0.918      0.109

Highest fraud F1: No resampling  (F1 = 0.717)
Saved plots/assignment_6/sampling_comparison.png


**Sampling comparison** (test set is still the real, imbalanced mix)

![Sampling comparison](../plots/assignment_6/sampling_comparison.png)


In [11]:
section("Part D: Which method, which metric, hypothesis test")

print("1. Which sampling method worked best?")
print(f"   By fraud F1, {best_name} wins ({scores.loc[best_name, 'f1']:.3f}).")
print("   The unsampled logistic regression already catches 63% of test frauds")
print("   at precision 0.83. Forcing a 50/50 training mix pushes recall to 0.92")
print("   but precision falls to about 0.04-0.06: thousands of false alarms.")
print("   F1 then collapses, so 'balanced training' is not automatically better.")
print("   Among the three samplers, random over-sampling has the best F1,")
print("   SMOTE is close behind, and under-sampling is worst because it also")
print("   throws away legitimate examples.")
print("   If the bank's goal is 'miss almost no fraud', use a sampler and then")
print("   raise the decision threshold until precision is acceptable.")
print("   SMOTE did not beat plain copying here: only 394 training frauds in")
print("   30 dimensions, so a synthetic neighbour is not new information.")
print()
print("2. Which metrics matter for fraud?")
print("   Recall first: a missed fraud (false negative) costs the bank real money.")
print("   Precision next: every false alarm is a payment stopped for review.")
print("   F1 is the single number that punishes a model that only does one of those.")
print("   Accuracy is the wrong headline. 99.8% is achievable by catching nothing.")
print()

# Hypothesis test as a feature check, fit on TRAIN only so the test set stays clean.
# Mann-Whitney U asks: does this feature tend to be larger for fraud than for
# legitimate payments? It does not assume a normal distribution. Amount is skewed,
# so a t-test on the raw dollars would be the wrong default.
# With ~200,000 training rows, a tiny shift is "significant". The p-value will be
# ~0 for almost every V-column. The useful number is the effect size
# (rank-biserial correlation): how strongly the feature separates the two classes.
legit_mask = y_train.to_numpy() == 0
fraud_mask = y_train.to_numpy() == 1
records = []
for j, col in enumerate(feature_cols):
    a = X_train_s[legit_mask, j]
    b = X_train_s[fraud_mask, j]
    # alternative='two-sided': we do not assume fraud is always larger.
    u_stat, p_value = stats.mannwhitneyu(b, a, alternative="two-sided")
    # Rank-biserial r is in [-1, 1]. 0 means the two classes are mixed together.
    r_rb = (2 * u_stat) / (a.size * b.size) - 1
    records.append({"feature": col, "p_value": p_value, "effect_r": r_rb, "abs_effect": abs(r_rb)})

tests = pd.DataFrame(records).sort_values("abs_effect", ascending=False)
print("3. Hypothesis testing for feature selection (train set, Mann-Whitney U)")
print("   Top 8 features by |effect size|, not by p-value:")
print(f"   {'feature':<10} {'p-value':>12} {'effect r':>10}")
print("   " + "-" * 34)
for _, row in tests.head(8).iterrows():
    print(f"   {row['feature']:<10} {row['p_value']:>12.2e} {row['effect_r']:>10.3f}")
n_sig = int((tests["p_value"] < 0.05).sum())
print(f"\n   Features with p < 0.05: {n_sig} of {len(tests)}")
print("   How this supports feature selection:")
print("   - Keep features whose class distributions actually differ (large |effect r|).")
print("   - Do not rank features by p-value alone. Here almost every p-value is tiny")
print("     because the sample is huge, even when the shift is too small to help a model.")
print("   - V14, V4, V12, V10 and the other large-|r| columns are the ones a")
print("     fraud model should be allowed to use. A feature with r near 0 can be dropped.")

top = tests.head(10).iloc[::-1]
plt.figure(figsize=(8, 5))
colors = ["crimson" if v > 0 else "steelblue" for v in top["effect_r"]]
plt.barh(top["feature"], top["effect_r"], color=colors)
plt.axvline(0, color="gray", lw=0.8)
plt.xlabel("Rank-biserial effect (fraud vs legitimate)")
plt.title("Features that separate fraud (train set)")
save_plot("hypothesis_effects.png")



  Part D: Which method, which metric, hypothesis test
1. Which sampling method worked best?
   By fraud F1, No resampling wins (0.717).
   The unsampled logistic regression already catches 63% of test frauds
   at precision 0.83. Forcing a 50/50 training mix pushes recall to 0.92
   but precision falls to about 0.04-0.06: thousands of false alarms.
   F1 then collapses, so 'balanced training' is not automatically better.
   Among the three samplers, random over-sampling has the best F1,
   SMOTE is close behind, and under-sampling is worst because it also
   throws away legitimate examples.
   If the bank's goal is 'miss almost no fraud', use a sampler and then
   raise the decision threshold until precision is acceptable.
   SMOTE did not beat plain copying here: only 394 training frauds in
   30 dimensions, so a synthetic neighbour is not new information.

2. Which metrics matter for fraud?
   Recall first: a missed fraud (false negative) costs the bank real money.
   Precision next

**Which features actually differ between fraud and legitimate payments**

![Hypothesis-test effect sizes](../plots/assignment_6/hypothesis_effects.png)


## Problem 2 — SMS spam vs ham

Each row is one text message labelled `ham` (normal) or `spam`. A Naive Bayes classifier treats the words as evidence and multiplies their probabilities. Two variants:

- **Bernoulli NB** cares only whether a word appears at least once (binary features).
- **Multinomial NB** cares how many times a word appears (count features).


In [12]:
section("Problem 2: SMS spam — Bernoulli NB vs Multinomial NB")

sms = pd.read_csv(DATA / "sms_spam.tsv", sep="\t", header=None, names=["label", "text"])
print(f"Messages: {len(sms):,}")
print(sms["label"].value_counts().to_string())
print("\nRaw example:")
print(" ", sms.loc[0, "text"][:140])

# Preprocessing, in the order the question asks for.
# 1. lowercase, so "Free" and "free" are the same word.
# 2. strip punctuation. "free!" and "free" would otherwise be different tokens.
# 3. drop English stop words (the, a, to, ...). They appear in both classes
#    and mostly add noise. sklearn's built-in list is used so we do not need NLTK.
punct_table = str.maketrans("", "", string.punctuation)

def clean_message(text):
    lowered = text.lower().translate(punct_table)
    tokens = [word for word in lowered.split() if word not in ENGLISH_STOP_WORDS and word.isalpha()]
    return " ".join(tokens)

sms["clean"] = sms["text"].map(clean_message)
print("\nCleaned example:")
print(" ", sms.loc[0, "clean"][:140])

# 1 = spam, so precision/recall below are "of the spam class".
y_sms = (sms["label"] == "spam").astype(int)
X_txt_train, X_txt_test, y_txt_train, y_txt_test = train_test_split(
    sms["clean"], y_sms, test_size=0.2, random_state=42, stratify=y_sms
)

# CountVectorizer builds a vocabulary from the TRAINING messages only,
# then counts how often each word appears. binary=True turns every count
# into 0/1, which is exactly the input Bernoulli NB is designed for.
# The default (binary=False) keeps the counts, which is what Multinomial NB expects.
binary_vec = CountVectorizer(binary=True)
count_vec = CountVectorizer(binary=False)
X_bin_train = binary_vec.fit_transform(X_txt_train)
X_bin_test = binary_vec.transform(X_txt_test)
X_cnt_train = count_vec.fit_transform(X_txt_train)
X_cnt_test = count_vec.transform(X_txt_test)
print(f"\nVocabulary size: {len(count_vec.vocabulary_):,} words")
print(f"Train messages: {X_cnt_train.shape[0]:,}    Test messages: {X_cnt_test.shape[0]:,}")

# alpha=1 is Laplace smoothing: pretend we have seen every word once.
# Without it, a word that never appeared in spam would give spam probability 0
# for the whole message, because Naive Bayes multiplies word probabilities.
bern = BernoulliNB(alpha=1.0)
multi = MultinomialNB(alpha=1.0)
bern.fit(X_bin_train, y_txt_train)
multi.fit(X_cnt_train, y_txt_train)

comparisons = []
for name, model, X_te in [
    ("Bernoulli NB (word present/absent)", bern, X_bin_test),
    ("Multinomial NB (word counts)", multi, X_cnt_test),
]:
    pred = model.predict(X_te)
    comparisons.append({
        "model": name,
        "accuracy": accuracy_score(y_txt_test, pred),
        "precision": precision_score(y_txt_test, pred, zero_division=0),
        "recall": recall_score(y_txt_test, pred, zero_division=0),
        "f1": f1_score(y_txt_test, pred, zero_division=0),
    })

comp = pd.DataFrame(comparisons)
print(f"\n{'Model':<40} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 84)
for _, row in comp.iterrows():
    print(
        f"{row['model']:<40} {100 * row['accuracy']:>9.2f}% {row['precision']:>10.3f} "
        f"{row['recall']:>10.3f} {row['f1']:>10.3f}"
    )

winner = comp.loc[comp["f1"].idxmax(), "model"]
print(f"\nBetter spam F1: {winner}")
print()
print("Why the two models want different features:")
print("  Bernoulli NB models each word as a yes/no event, P(word appears | class).")
print("  Repeating 'free free free' is the same evidence as writing 'free' once.")
print("  That matches a binary bag-of-words, and short SMS where presence matters")
print("  more than repetition.")
print("  Multinomial NB models the count of each word, as if the message were a")
print("  bag of tokens drawn with replacement. A word used five times is five")
print("  pieces of evidence. That matches CountVectorizer's raw counts")
print("  (or TF-IDF, which is a weighted count).")
print("  On SMS the counts are usually 0 or 1 anyway, so the two models often")
print("  land close. Multinomial still has the edge when a spam word is repeated.")



  Problem 2: SMS spam — Bernoulli NB vs Multinomial NB
Messages: 5,572
label
ham     4825
spam     747

Raw example:
  Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Cleaned example:
  jurong point crazy available bugis n great world la e buffet cine got amore wat

Vocabulary size: 6,964 words
Train messages: 4,457    Test messages: 1,115

Model                                      Accuracy  Precision     Recall         F1
------------------------------------------------------------------------------------
Bernoulli NB (word present/absent)           96.59%      0.991      0.752      0.855
Multinomial NB (word counts)                 97.76%      0.949      0.879      0.913

Better spam F1: Multinomial NB (word counts)

Why the two models want different features:
  Bernoulli NB models each word as a yes/no event, P(word appears | class).
  Repeating 'free free free' is the same evidence as writing 'free' once.
  That mat